# 📰 News Topic Classifier Using BERT

**Objective:** Fine-tune a transformer model (`bert-base-uncased`) to classify news headlines into one of four topic categories using the AG News dataset, then deploy the model as a live interactive demo.

**Dataset:** [AG News](https://huggingface.co/datasets/fancyzhx/ag_news) — 4 balanced classes: `World`, `Sports`, `Business`, `Sci/Tech`. (Note: this dataset was recently moved on the Hub from the short name `ag_news` to `fancyzhx/ag_news` — we use the current path below.)

**Pipeline covered in this notebook:**
1. Load and explore the AG News dataset
2. Tokenize and preprocess text with the BERT tokenizer
3. Fine-tune `bert-base-uncased` for sequence classification using Hugging Face `Trainer`
4. Evaluate using Accuracy and F1-score (plus a full classification report + confusion matrix)
5. Save the fine-tuned model
6. Deploy a live demo with **Gradio** (runs directly in this notebook) and a **Streamlit** app (runs as a standalone script)

> 💡 **Compute tip:** Fine-tuning BERT is much faster on a GPU. If you're running this in Google Colab, go to `Runtime > Change runtime type > GPU` (the free tier is enough for this task). On CPU it will still work but will take considerably longer — the notebook includes a toggle to train on a smaller subset of the data for quick experimentation.

## 1. Install Dependencies

We need `transformers` (BERT model + training utilities), `datasets` (to load AG News), `evaluate` (metrics), `accelerate` (efficient training), `scikit-learn` / `matplotlib` / `seaborn` (evaluation report + confusion matrix), and `gradio` (deployment).

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn matplotlib seaborn gradio

## 2. Import Libraries

Standard imports for data handling, PyTorch, Hugging Face Transformers/Datasets/Evaluate, and plotting.

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import evaluate
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 3. Load the AG News Dataset

The AG News dataset is loaded directly from the Hugging Face Hub. It comes with a pre-defined `train` split (120,000 examples) and `test` split (7,600 examples), each with a `text` field (the headline + short description) and a `label` field (0–3).

In [ ]:
# Note: this dataset was renamed on the Hub from "ag_news" to "fancyzhx/ag_news".
# Loading the old short name can raise an HfUriError on newer huggingface_hub versions,
# so we use the current repo id directly.
dataset = load_dataset("fancyzhx/ag_news")
dataset

## 4. Explore the Dataset

Let's look at the class names and a sample record to understand the data format before preprocessing.

In [ ]:
label_names = dataset["train"].features["label"].names
print("Label names:", label_names)

print("\nTrain size:", len(dataset["train"]))
print("Test size:", len(dataset["test"]))

print("\nSample record:")
print(dataset["train"][0])

## 5. (Optional) Use a Subset for Faster Training

Fine-tuning on the full 120k training examples works well but can take a long time on a single GPU (and a very long time on CPU). Set `USE_SUBSET = True` to train quickly on a smaller sample first (e.g. to verify the pipeline works), then set it to `False` and re-run for the full, higher-accuracy training run.

In [ ]:
USE_SUBSET = True

TRAIN_SUBSET_SIZE = 8000
TEST_SUBSET_SIZE = 2000

if USE_SUBSET:
    train_dataset = dataset["train"].shuffle(seed=42).select(range(TRAIN_SUBSET_SIZE))
    test_dataset = dataset["test"].shuffle(seed=42).select(range(TEST_SUBSET_SIZE))
else:
    train_dataset = dataset["train"]
    test_dataset = dataset["test"]

print(f"Training on {len(train_dataset)} examples, evaluating on {len(test_dataset)} examples")

## 6. Load the BERT Tokenizer

We use the tokenizer that matches `bert-base-uncased` so text is split into WordPiece tokens exactly the way the pretrained model expects.

In [ ]:
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

## 7. Tokenize the Dataset

We define a preprocessing function that pads/truncates every headline to a fixed length (128 tokens is more than enough for news headlines) and apply it to both splits with `.map()`, which processes the data in batches for speed.

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

## 8. Format Data for PyTorch / Trainer

The `Trainer` API expects a `labels` column (not `label`) and PyTorch tensors. We rename the column and set the dataset format accordingly, keeping only the columns the model needs.

In [ ]:
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

columns_to_keep = ["input_ids", "attention_mask", "labels"]
tokenized_train.set_format(type="torch", columns=columns_to_keep)
tokenized_test.set_format(type="torch", columns=columns_to_keep)

## 9. Load Pretrained BERT for Sequence Classification

`AutoModelForSequenceClassification` loads `bert-base-uncased` and attaches a fresh classification head with 4 output units (one per AG News category). We also attach human-readable label names to the model config so predictions are easy to interpret later.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_names)
)

model.config.id2label = {i: name for i, name in enumerate(label_names)}
model.config.label2id = {name: i for i, name in enumerate(label_names)}

model.to(device)

## 10. Define Evaluation Metrics

We use the `evaluate` library to compute **Accuracy** and **weighted F1-score** (weighted accounts for any class imbalance, though AG News is balanced). This function is passed to the `Trainer` and is run automatically at every evaluation step.

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

## 11. Define Training Arguments

Key hyperparameters for fine-tuning: a small learning rate (typical for fine-tuning pretrained transformers), a modest number of epochs (BERT converges quickly and overfits easily on text classification), and evaluation/checkpointing after every epoch so we can track progress.

> Note: depending on your installed `transformers` version, this argument is called `eval_strategy` (newer) or `evaluation_strategy` (older). If you get a `TypeError` about an unexpected keyword, just rename it.

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert-ag-news-results",
    eval_strategy="epoch",          # use "evaluation_strategy" on older transformers versions
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=50,
    report_to="none"
)

## 12. Initialize the Trainer

The `Trainer` class wraps the training loop, evaluation loop, checkpointing, and metric computation for us.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

## 13. Fine-tune the Model

This runs the actual training loop. With the default subset (8,000 train examples) and 3 epochs, this should take a few minutes on a GPU. Watch the `eval_f1` and `eval_accuracy` columns printed after each epoch.

In [ ]:
trainer.train()

## 14. Evaluate the Model

Run a final evaluation pass on the held-out test set and print overall Accuracy and F1-score.

In [ ]:
eval_results = trainer.evaluate()
print(eval_results)

## 15. Detailed Classification Report & Confusion Matrix

Beyond the aggregate accuracy/F1, it's useful to see per-class precision/recall/F1 and where the model gets confused (e.g. `Business` vs `Sci/Tech` headlines can overlap in vocabulary).

In [ ]:
predictions_output = trainer.predict(tokenized_test)
y_pred = np.argmax(predictions_output.predictions, axis=-1)
y_true = predictions_output.label_ids

print(classification_report(y_true, y_pred, target_names=label_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - AG News Topic Classification")
plt.tight_layout()
plt.show()

## 16. Save the Fine-tuned Model

Save both the model weights and tokenizer so they can be reloaded later (e.g. by the deployment app) without retraining.

In [ ]:
save_path = "./bert-ag-news-finetuned"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

## 17. Quick Inference Test

Load the saved model into a Hugging Face `pipeline` for easy inference and try it on a few new, unseen headlines.

In [ ]:
classifier = pipeline(
    "text-classification",
    model=save_path,
    tokenizer=save_path,
    device=0 if device == "cuda" else -1
)

sample_headlines = [
    "Apple unveils new AI chip for next-generation iPhones",
    "Manchester United wins dramatic match in final minute",
    "Federal Reserve raises interest rates amid inflation concerns",
    "NASA's new telescope captures stunning images of distant galaxy"
]

for headline in sample_headlines:
    result = classifier(headline)[0]
    print(f"Headline: {headline}")
    print(f"Predicted Topic: {result['label']}  (confidence: {result['score']:.2%})\n")

## 18. Deploy with Gradio (runs live in this notebook)

Gradio can launch an interactive web demo directly from a notebook cell — perfect for quickly sharing a working prototype. Set `share=True` to get a temporary public link you can send to anyone (e.g. for a course submission or portfolio demo).

In [ ]:
import gradio as gr

def classify_headline(headline):
    if not headline.strip():
        return "Please enter a headline."
    result = classifier(headline)[0]
    return f"{result['label']}  (confidence: {result['score']:.2%})"

demo = gr.Interface(
    fn=classify_headline,
    inputs=gr.Textbox(lines=2, placeholder="Enter a news headline..."),
    outputs=gr.Textbox(label="Predicted Topic"),
    title="News Topic Classifier (BERT fine-tuned on AG News)",
    description="Enter a news headline and the model will classify it into one of four categories: World, Sports, Business, or Sci/Tech.",
    examples=sample_headlines
)

demo.launch(share=True)

## 19. (Alternative) Deploy with Streamlit

Streamlit apps run as standalone scripts (not inside notebook cells), so the cell below writes a ready-to-run `app.py` file to disk using the `%%writefile` magic. To launch it, open a terminal in the same folder as your saved model and run:

```bash
streamlit run app.py
```

This is a great option if you want a more polished, standalone demo (or to deploy for free on [Hugging Face Spaces](https://huggingface.co/spaces) or [Streamlit Community Cloud](https://streamlit.io/cloud)).

In [ ]:
%%writefile app.py
import streamlit as st
from transformers import pipeline

st.set_page_config(page_title="News Topic Classifier", page_icon="📰")

st.title("📰 News Topic Classifier")
st.write(
    "Fine-tuned BERT model trained on the AG News dataset. "
    "Classifies headlines into World, Sports, Business, or Sci/Tech."
)

@st.cache_resource
def load_classifier():
    return pipeline(
        "text-classification",
        model="./bert-ag-news-finetuned",
        tokenizer="./bert-ag-news-finetuned"
    )

classifier = load_classifier()

headline = st.text_area("Enter a news headline:", height=100)

if st.button("Classify"):
    if headline.strip():
        result = classifier(headline)[0]
        st.success(f"**Predicted Topic:** {result['label']}")
        st.write(f"Confidence: {result['score']:.2%}")
    else:
        st.warning("Please enter a headline first.")

## 20. Summary & Next Steps

**What this notebook covered:**
- Loading and exploring the AG News dataset via Hugging Face `datasets`
- Tokenizing text with the `bert-base-uncased` tokenizer
- Fine-tuning BERT for 4-class sequence classification with the `Trainer` API
- Evaluating with Accuracy, weighted F1-score, a full classification report, and a confusion matrix
- Deploying a live demo with both **Gradio** (in-notebook) and **Streamlit** (standalone app)

**Ideas to extend this for your assignment / portfolio:**
- Set `USE_SUBSET = False` and train on the full 120k examples for a stronger final model (best done on a GPU)
- Try `distilbert-base-uncased` for a lighter, faster model with a small accuracy trade-off — good talking point on the accuracy/latency trade-off
- Add a learning-rate scheduler sweep or try 2 vs 3 vs 4 epochs and compare F1
- Push the fine-tuned model to the Hugging Face Hub (`trainer.push_to_hub()`) so it can be loaded from anywhere without re-uploading files, and host the Gradio/Streamlit app for free on Hugging Face Spaces
- Add a "confidence too low" fallback in the app UI (e.g. flag predictions below a threshold as uncertain)